# 01 Bronze: ingest the raw CSV

**Goal:** land the raw file in a Delta table exactly as it arrives, so we can always reprocess from it.

- Source: `air_pollution.csv` uploaded to a Unity Catalog volume
- Target: `bronze_air_pollution` (all columns kept as strings, plus audit columns)
- The only change: the column `date` is renamed `reading_date` to avoid clashing with the SQL keyword

In [0]:
CATALOG = "workspace"      
SCHEMA = "air_quality"
VOLUME = "raw"
FILE_NAME = "air_pollution.csv"

RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{FILE_NAME}"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_air_pollution"

In [0]:
# Create the schema and the volume once.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

DataFrame[]

**Manual step (first run only):** upload `data/air_pollution.csv` to the volume.
In the left sidebar go to *Catalog* > `workspace` > `air_quality` > `raw` > *Upload to this volume*.

In [0]:
from pyspark.sql import functions as F

try:
    dbutils.fs.ls(RAW_PATH)
except Exception:
    raise FileNotFoundError(
        f"{RAW_PATH} not found. Upload data/air_pollution.csv to the '{VOLUME}' volume first."
    )

raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)          # keep everything as string in bronze
    .csv(RAW_PATH)
    .withColumnRenamed("date", "reading_date")
)

bronze = (
    raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit(FILE_NAME))
)

# Full reload of the same file gives the same result, so re-running is safe (idempotent).
(
    bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(BRONZE_TABLE)
)

In [0]:
print("Rows in bronze:", spark.table(BRONZE_TABLE).count())
display(spark.table(BRONZE_TABLE).limit(10))

Rows in bronze: 576


reading_date,pollutant,concentration,_ingested_at,_source_file
2017-01-01,CO,0.561,2026-10-22T06:01:59.057Z,air_pollution.csv
2017-02-01,CO,0.53,2026-10-22T06:01:59.057Z,air_pollution.csv
2017-03-01,CO,0.589,2026-10-22T06:01:59.057Z,air_pollution.csv
2017-04-01,CO,0.662,2026-10-22T06:01:59.057Z,air_pollution.csv
2017-05-01,CO,null,2026-10-22T06:01:59.057Z,air_pollution.csv
2017-06-01,CO,null,2026-10-22T06:01:59.057Z,air_pollution.csv
2017-07-01,CO,0.716,2026-10-22T06:01:59.057Z,air_pollution.csv
2017-08-01,CO,0.661,2026-10-22T06:01:59.057Z,air_pollution.csv
2017-09-01,CO,0.682,2026-10-22T06:01:59.057Z,air_pollution.csv
2017-10-01,CO,0.68,2026-10-22T06:01:59.057Z,air_pollution.csv
